Use the model to help make decision whether to spend now or 7 days down the line.

In [ ]:
""" extract only the future forecast """

today = prophet_df["ds"].max()
future_forecast = forecast[forecast["ds"] > today][["ds", "yhat", "yhat_lower", "yhat_upper"]]

future_forecast.head()

In [ ]:
""" create a simple cost estimator """

def estimate_inr(eur_amount, target_date, forecast_df):

  row = forecast_df[forecast_df["ds"] == target_date]

  if row.empty:
    raise ValueError("Date not in forecast range")

  rate = row["yhat"].values[0]
  lower = row["yhat_lower"].values[0]
  upper = row["yhat_upper"].values[0]

  return {
      "predicted_inr": eur_amount * rate,
      "best_case": eur_amount * lower,
      "worst_case": eur_amount * upper
  }

In [ ]:
""" calculate the cost """

from datetime import timedelta

eur_amount = int(input("Enter euro amount that you are planning to spend 7 days from today: "))

today_date = future_forecast["ds"].iloc[0]
next_week_date = today_date + timedelta(days=7)

today_cost = estimate_inr(eur_amount, today_date, future_forecast)
next_week_cost = estimate_inr(eur_amount, next_week_date, future_forecast)

print("\nToday: ", today_cost)
print("\nNext Week: ", next_week_cost)


In [ ]:
""" build a comparison table """

comparison_df = pd.DataFrame(
    [
        {
            "Date": today_date.date(),
            "Estimated INR": today_cost["predicted_inr"]
        },
        {
            "Date": next_week_date.date(),
            "Estimated INR": next_week_cost["predicted_inr"]
        }
    ]
)

comparison_df

In [ ]:
""" plot the graph of Spend Now vs Spend Later """

plt.figure()
plt.bar(
    comparison_df["Date"].astype(str),
    comparison_df["Estimated INR"]
)
plt.title("Estimated INR Spend Now vs Spend Later")
plt.xlabel("Date")
plt.show()

In [ ]:
""" include forex markup fees and GST """

## adding charges for today
forex_markup_fees = 0.035 * today_cost["predicted_inr"]
gst = 0.18 * today_cost["predicted_inr"]

total_cost_today = forex_markup_fees + gst + today_cost["predicted_inr"]

## adding charges for next week
forex_markup_fees = 0.035 * next_week_cost["predicted_inr"]
gst = 0.18 * next_week_cost["predicted_inr"]

total_cost_next_week = forex_markup_fees + gst + next_week_cost["predicted_inr"]

print("\nIncluding all the fees, we get - ")
print("\tTotal today cost: ", total_cost_today)
print("\tTotal next week cost: ", total_cost_next_week)

if total_cost_today > total_cost_next_week:
  print("\nI think you should wait, most likely the Euro will go down!")
else:
  print("\nI think you should spend now, most likely the Euro will go up!")